# Find duplicate records

**The job.** One customer list. The same people are in it more than once, spelled
differently. Find the pairs.

The naive version compares every record with every other one. For 5,000 records
that is 12.5 million comparisons. The fix is **blocking**: only compare records
that share something cheap, like the first letter of a surname plus a postcode.

Blocking and scoring are independent, and they meet at the decision. Another
diamond.

**In:** a list of customers with duplicates in it.
**Out:** matched pairs, and the ones too close to call.
**Files:** matches.jsonl, review.jsonl.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


In [2]:
PEOPLE = [
    ("1",  "Jonathan Smith",   "jsmith@example.com",  "SW1A 1AA"),
    ("2",  "Jon Smith",        "jsmith@example.com",  "SW1A 1AA"),
    ("3",  "J. Smith",         "j.smith@example.com", "SW1A 1AA"),
    ("4",  "Priya Raman",      "praman@example.com",  "M1 4BT"),
    ("5",  "Priya Ramen",      "praman@example.com",  "M1 4BT"),
    ("6",  "Alex Okafor",      "aokafor@example.com", "EH1 2NG"),
    ("7",  "Alexandra Okafor", "a.okafor@example.com","EH1 2NG"),
    ("8",  "Wei Zhang",        "wzhang@example.com",  "LS1 5AA"),
    ("9",  "Wei Zhang",        "wei.zhang@work.com",  "BS1 6TP"),
    ("10", "Marta Nowak",      "mnowak@example.com",  "CF10 1EP"),
]
records = [{"id": i, "name": n, "email": e, "postcode": p} for i, n, e, p in PEOPLE]
print(f"{len(records)} records")
for r in records[:4]:
    print(" ", r)

10 records
  {'id': '1', 'name': 'Jonathan Smith', 'email': 'jsmith@example.com', 'postcode': 'SW1A 1AA'}
  {'id': '2', 'name': 'Jon Smith', 'email': 'jsmith@example.com', 'postcode': 'SW1A 1AA'}
  {'id': '3', 'name': 'J. Smith', 'email': 'j.smith@example.com', 'postcode': 'SW1A 1AA'}
  {'id': '4', 'name': 'Priya Raman', 'email': 'praman@example.com', 'postcode': 'M1 4BT'}


In [3]:
nodes = [
    node("load.people",  "read",  [],                    [("out", "Records")]),
    node("block.key",    "block", [("in", "Records")],   [("out", "Blocks")]),
    node("score.name",   "score", [("in", "Blocks")],    [("out", "Scores")]),
    node("score.contact","score", [("in", "Blocks")],    [("out", "Scores")]),
    node("decide.pairs", "decide",[("name", "Scores"), ("contact", "Scores")],
         [("matched", "Pairs"), ("review", "Pairs")]),
    node("write.pairs",  "write", [("matched", "Pairs"), ("review", "Pairs")],
         [("out", "Receipt")], effects=("file.write",)),
]

stages = [
    stage("load",    "Load the list",     [],                  [("out", "Records")], "read",  ["load.people"]),
    stage("block",   "Group cheaply",     [("in", "Records")], [("out", "Blocks")],  "block", ["block.key"]),
    stage("name",    "Compare names",     [("in", "Blocks")],  [("out", "Scores")],  "score", ["score.name"]),
    stage("contact", "Compare contacts",  [("in", "Blocks")],  [("out", "Scores")],  "score", ["score.contact"]),
    stage("decide",  "Decide each pair",  [("name", "Scores"), ("contact", "Scores")],
          [("matched", "Pairs"), ("review", "Pairs")], "decide", ["decide.pairs"]),
    stage("write",   "Write both",        [("matched", "Pairs"), ("review", "Pairs")],
          [("out", "Receipt")], "write", ["write.pairs"]),
]

edges = [Edge("load", "block"), Edge("block", "name"), Edge("block", "contact"),
         Edge("name", "decide", to_port="name"),
         Edge("contact", "decide", to_port="contact"),
         Edge("decide", "write", from_port="matched", to_port="matched"),
         Edge("decide", "write", from_port="review", to_port="review")]

bench = build("Find duplicate records",
              "Find the same person listed twice, without comparing everything to everything.",
              stages, nodes, edges)
print("layers:", bench.layers())

problems: ["stage 'name' omits 1 compatible candidate(s), e.g. 'score.contact' — a stage must show everything that could perform it", "stage 'contact' omits 1 compatible candidate(s), e.g. 'score.name' — a stage must show everything that could perform it"]
layers: [['load'], ['block'], ['name', 'contact'], ['decide'], ['write']]


In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg31804888-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load the list</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Group cheaply</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Compare names</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Compare contacts</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Decide each pair</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Write both</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><text x="678.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">name</text><path d="M666,182.0 C678.0,182.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><text x="678.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">contact</text><path d="M876,141.0 C888.0,141.0 888.0,141.0 900,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><text x="888.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">matched</text><path d="M876,141.0 C888.0,141.0 888.0,141.0 900,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg31804888-arrow)"/><text x="888.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">review</text></svg>', title='Find duplicate records — shape', note='5 layers, widest 2. Boxes in the same layer are independent and may run together; every

In [5]:
import itertools
from difflib import SequenceMatcher

def load_people():
    return records

def block_key(**kw):
    """Only compare people who share a postcode. Cheap, and it cuts the work."""
    blocks = {}
    for record in kw["in"]:
        blocks.setdefault(record["postcode"], []).append(record)
    pairs = [(a, b) for group in blocks.values()
             for a, b in itertools.combinations(group, 2)]
    everything = len(kw["in"]) * (len(kw["in"]) - 1) // 2
    return {"pairs": pairs, "compared": len(pairs), "without_blocking": everything}

def score_name(**kw):
    return {f"{a['id']}-{b['id']}": SequenceMatcher(None, a["name"].lower(),
                                                    b["name"].lower()).ratio()
            for a, b in kw["in"]["pairs"]}

def score_contact(**kw):
    """Same email is strong evidence. Same domain is weak evidence."""
    out = {}
    for a, b in kw["in"]["pairs"]:
        if a["email"] == b["email"]:
            out[f"{a['id']}-{b['id']}"] = 1.0
        elif a["email"].split("@")[1] == b["email"].split("@")[1]:
            out[f"{a['id']}-{b['id']}"] = 0.4
        else:
            out[f"{a['id']}-{b['id']}"] = 0.0
    return out

def decide_pairs(**kw):
    """Two independent scores, one decision, and an honest middle."""
    matched, review = [], []
    by_id = {r["id"]: r for r in records}
    for key, name_score in kw["name"].items():
        contact_score = kw["contact"][key]
        combined = 0.5 * name_score + 0.5 * contact_score
        left, right = key.split("-")
        row = {"pair": key, "a": by_id[left]["name"], "b": by_id[right]["name"],
               "name": round(name_score, 3), "contact": contact_score,
               "combined": round(combined, 3)}
        if combined >= 0.75:
            matched.append(row)
        elif combined >= 0.45:
            review.append(row)
    return {"matched": matched, "review": review}

def write_pairs(workspace, **kw):
    (workspace / "matches.jsonl").write_text(
        "\n".join(json.dumps(r) for r in kw["matched"]))
    (workspace / "review.jsonl").write_text(
        "\n".join(json.dumps(r) for r in kw["review"]))
    return {"matched": len(kw["matched"]), "review": len(kw["review"])}

runtime = execute.Runtime({
    "load.people": load_people, "block.key": block_key, "score.name": score_name,
    "score.contact": score_contact, "decide.pairs": decide_pairs,
    "write.pairs": write_pairs})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})
run = execute.run(plan, runtime, workspace=WORK, workers=2)
print(run.text())

plan plan:e6fb6be22162d22c61752…
6 steps in 0.002s — ok
  ok   load             0.000s  load.people
  ok   block            0.000s  block.key
  ok   name             0.000s  score.name
  ok   contact          0.000s  score.contact
  ok   decide           0.000s  decide.pairs
  ok   write            0.000s  write.pairs  [file.write]
  file /home/username/code_projects/repos/browsergraph/notebooks/work/matches.jsonl  212 bytes  sha256:461556b88…
  file /home/username/code_projects/repos/browsergraph/notebooks/work/review.jsonl  317 bytes  sha256:bbfd89349…


## How much work blocking saved

In [6]:
blocks = run.output("block")
print(f"compared {blocks['compared']} pairs")
print(f"without blocking it would be {blocks['without_blocking']}")
print(f"saved {1 - blocks['compared'] / blocks['without_blocking']:.0%} of the work")

compared 5 pairs
without blocking it would be 45
saved 89% of the work


## The pairs

In [7]:
matched = run.values[("decide", "matched")]
review = run.values[("decide", "review")]

print(f"{'pair':<7}{'a':<20}{'b':<20}{'name':>7}{'contact':>9}{'combined':>10}")
for row in matched:
    print(f"{row['pair']:<7}{row['a']:<20}{row['b']:<20}"
          f"{row['name']:>7}{row['contact']:>9}{row['combined']:>10}   MATCH")
for row in review:
    print(f"{row['pair']:<7}{row['a']:<20}{row['b']:<20}"
          f"{row['name']:>7}{row['contact']:>9}{row['combined']:>10}   review")

pair   a                   b                      name  contact  combined
1-2    Jonathan Smith      Jon Smith             0.783      1.0     0.891   MATCH
4-5    Priya Raman         Priya Ramen           0.909      1.0     0.955   MATCH
1-3    Jonathan Smith      J. Smith              0.636      0.4     0.518   review
2-3    Jon Smith           J. Smith              0.824      0.4     0.612   review
6-7    Alex Okafor         Alexandra Okafor      0.815      0.4     0.607   review


Note what is **not** here. Wei Zhang appears twice with the same name and
different postcodes, so blocking never compared them. That is the trade
blocking makes: less work, and some pairs you will never see.

Saying that out loud matters. A dedupe run that reports "3 duplicates found"
without mentioning what it never looked at is telling you half the answer.

In [8]:
print("files written:")
for art in run.artifacts:
    print(f"  {art.path:<34} {art.bytes:>8,} bytes  {art.digest[:18]}…")

files written:
  /home/username/code_projects/repos/browsergraph/notebooks/work/matches.jsonl      212 bytes  sha256:461556b88d4…
  /home/username/code_projects/repos/browsergraph/notebooks/work/review.jsonl      317 bytes  sha256:bbfd8934905…


In [9]:
# Where the time actually went. Colour carries the outcome: a step that was
# cached, one skipped by a branch and one that fell back to another candidate
# all "succeeded", and they are not the same thing.
viz.timeline(run, title='blocking, then comparing')

Figure(svg='<svg viewBox="0 0 1000 306" width="1000" height="306" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.001s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">load</text><rect x="316.6" y="77" width="31.5" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>load.people — ran, 0.1ms</title></rect><text x="357.1" y="91" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">block</text><rect x="358.7" y="107" width="26.9" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>block.key — ran, 0.1ms</title></rect><text x="394.6" y="121" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">name</text><rect x="506.9" y="137" width="86.6" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>score.name — ran, 0.2ms</title></rect><text x="602.4" y="151" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">contact</text><rect x="631.9" y="167" width="24.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>score.contact — ran, 0.1ms</title></rect><text x="665.2" y="181" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="210" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">decide</text><rect x="702.4" y="197" width="35.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>decide.pairs — ran, 0.1ms</title></rect><text x="746.7" y="211" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="240" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">write</text><rect x="746.1" y="227" width="123.9" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>write.pairs — ran, 0.3ms</title></rect><text x="879.0" y="241" font-size="10" fill="#68737f">0ms · ran</text><rect x="190" y="274" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="283" font-size="10" fill="#68737f">failed</text> <rect x="286" y="274" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="283" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="274" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="283" font-size="10" fill="#68737f">cached</text> <rect x="478" y="274" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="283" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="274" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="283" font-size="10" fill="#68737f">ran</text></svg>', title='blocking, then comparing', note='Bars are placed at the time each step began.', width=1000, height=306)